# Phase 5 — Shopper Universe Flow Visualization

**Purpose:** Visualize the complete shopper journey through sizes — from trial to repeat to lapse — with ASP annotations to identify which flow and price point to fix.

**This is NOT the standard `08_shopper_flow.ipynb` template.** It is a custom funnel + alluvial analysis built specifically for the sizing/pricing strategy question.

| Step | Description |
|------|-------------|
| 5-1 | Size-level funnel: Category Trial → Sub-brand Trial → Repeat → Lapse (per size, absolute numbers) |
| 5-2 | Alluvial/Sankey: trial size → repeat size → lapse destination (sub-brand + size) |
| 5-3 | ASP-annotated flow: overlay prevailing ASP at each node to connect price to flow volume |

**Created:** 2026-02-19

---
## 0. Imports & Connection

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
from dotenv import load_dotenv
import databricks.sql as sql
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

def _find_japanese_font():
    for name in ['MS Gothic', 'MS PGothic', 'Yu Gothic', 'Meiryo', 'IPAexGothic']:
        if name in {f.name for f in fm.fontManager.ttflist}:
            return name
    return None
_jp_font = _find_japanese_font()
if _jp_font:
    plt.rcParams['font.family'] = _jp_font
    print(f'✅ Japanese font: {_jp_font}')

load_dotenv(dotenv_path='../../.env')
DATABRICKS_HOST      = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN     = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')
assert all([DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_HTTP_PATH]), 'Missing .env credentials'
print('✅ Credentials loaded')

def execute_query(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

✅ Japanese font: MS Gothic
✅ Credentials loaded


---
## 1. Parameters

In [2]:
ARIEL_GEL   = 'ｱﾘｴｰﾙｼﾞｪﾙ'
ATTACK_EX   = 'ｱﾀｯｸ抗菌EX'
SUB_CAT     = '洗濯洗剤'
CATEGORY    = 'Laundry'

LOOKBACK_START = '2024-01-01'
ANALYSIS_START = '2025-01-01'
ANALYSIS_END   = '2026-01-31'
RENEWAL_MONTH  = '2025-05-01'
LAPSE_WINDOW_DAYS = 180
LATEST_COHORT_END = '2025-07-31'

RETAILER_CODES = [
    'cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009',
    'cds_8010', 'cds_8011', 'cds_8012', 'cds_8013',
]
RETAILER_IN = ', '.join(f"'{c}'" for c in RETAILER_CODES)

print(f'📋 Flow visualization: {ANALYSIS_START} → {ANALYSIS_END}')

📋 Flow visualization: 2025-01-01 → 2026-01-31


---
## 2. Build Complete Shopper Journey Dataset

One unified query that tracks each shopper's journey:
- Category trial OR existing category buyer
- Sub-brand trial (first Ariel Gel purchase)
- Entry size
- Repeat event (if any) and repeat size
- Lapse event and destination

In [4]:
# ── Extended-timeout helper for long-running queries ──────────────────
def execute_query_long(query: str) -> pd.DataFrame:
    """Like execute_query but with 1-hour retry timeout."""
    with sql.connect(
        server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
        access_token=DATABRICKS_TOKEN,
        _retry_stop_after_attempts_duration=3600
    ) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

# ── Simplified journey query (removed cat_first CTE, narrowed date range) ─
journey_query = f"""
WITH all_ariel AS (
    SELECT
        idpos.shopper_key,
        prod.jp_segment_4_name AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS asp
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    INNER JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name = '{ARIEL_GEL}'
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3
),
-- Sub-brand first purchase (trial event) within analysis window
sb_first AS (
    SELECT shopper_key, MIN(purchase_date) AS trial_date
    FROM all_ariel
    GROUP BY 1
),
-- Trial shoppers with entry size
trial_shoppers AS (
    SELECT
        sf.shopper_key,
        sf.trial_date,
        aa.size_code AS trial_size,
        aa.asp       AS trial_asp
    FROM sb_first sf
    INNER JOIN all_ariel aa ON sf.shopper_key = aa.shopper_key AND sf.trial_date = aa.purchase_date
    WHERE sf.trial_date BETWEEN '{ANALYSIS_START}' AND '{LATEST_COHORT_END}'
),
-- Find first repeat within 6 months
with_repeat AS (
    SELECT
        ts.*,
        aa.purchase_date AS repeat_date,
        aa.size_code     AS repeat_size,
        aa.asp           AS repeat_asp,
        ROW_NUMBER() OVER (PARTITION BY ts.shopper_key ORDER BY aa.purchase_date) AS rn
    FROM trial_shoppers ts
    LEFT JOIN all_ariel aa
           ON ts.shopper_key = aa.shopper_key
          AND aa.purchase_date > ts.trial_date
          AND aa.purchase_date <= DATE_ADD(ts.trial_date, {LAPSE_WINDOW_DAYS})
)
SELECT
    shopper_key,
    'Trial' AS cat_status,
    trial_date,
    trial_size,
    trial_asp,
    repeat_date,
    repeat_size,
    repeat_asp,
    CASE WHEN repeat_date IS NOT NULL THEN 'Repeat' ELSE 'Lapse' END AS outcome
FROM with_repeat
WHERE rn = 1 OR repeat_date IS NULL
"""

print('⏳ Building complete shopper journey dataset (simplified)...', flush=True)
df_journey = execute_query_long(journey_query)
df_journey['trial_date'] = pd.to_datetime(df_journey['trial_date'])
df_journey['repeat_date'] = pd.to_datetime(df_journey['repeat_date'])
for col in ['trial_asp', 'repeat_asp']:
    df_journey[col] = pd.to_numeric(df_journey[col])

print(f'✅ {len(df_journey):,} shopper journeys loaded')
print(f'   Repeat: {(df_journey["outcome"]=="Repeat").sum():,}')
print(f'   Lapse:  {(df_journey["outcome"]=="Lapse").sum():,}')

⏳ Building complete shopper journey dataset (simplified)...


HTTP request error: 'NoneType' object has no attribute 'request'


✅ 3,145,424 shopper journeys loaded
   Repeat: 1,581,414
   Lapse:  1,564,010


In [5]:
# ── Load lapse destinations from Phase 4 export (avoid re-querying) ───
# Phase 4 already computed the full lapse destination analysis
p4_flow = pd.read_excel('phase4_lapse_analysis.xlsx', sheet_name='Flow_Detail')
p4_dest = pd.read_excel('phase4_lapse_analysis.xlsx', sheet_name='Destination_Summary')

print(f'✅ Loaded Phase 4 lapse destination data')
print(f'   Flow detail rows: {len(p4_flow):,}')
print(f'   Destination summary rows: {len(p4_dest):,}')
print(f'   Top 5 destinations:')
print(p4_dest.head().to_string(index=False))

✅ Loaded Phase 4 lapse destination data
   Flow detail rows: 895
   Destination summary rows: 168
   Top 5 destinations:
next_sub_brand     next_size  shoppers  share_%
      ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ    141503      7.9
      ｱﾀｯｸ抗菌EX         詰替超特大     87686      4.9
      ｱﾀｯｸ抗菌EX  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     81852      4.6
      ｱﾀｯｸ抗菌EX   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ     58705      3.3
      ｱﾀｯｸ抗菌EX          本体通常     43467      2.4


---
## 3. Step 5-1: Size-Level Funnel — Category Trial → Sub-brand Trial → Repeat → Lapse

In [6]:
# ── Build funnel data per trial entry size ────────────────────────────
funnel_data = df_journey.groupby('trial_size').agg(
    total_trial     = ('shopper_key', 'nunique'),
    repeat_shoppers = ('outcome', lambda x: (x == 'Repeat').sum()),
    lapse_shoppers  = ('outcome', lambda x: (x == 'Lapse').sum()),
    avg_trial_asp   = ('trial_asp', 'mean'),
).reset_index()

funnel_data['repeat_rate_%'] = (funnel_data['repeat_shoppers'] / funnel_data['total_trial'] * 100).round(1)
funnel_data['lapse_rate_%']  = (funnel_data['lapse_shoppers'] / funnel_data['total_trial'] * 100).round(1)

print('Funnel Summary per Entry Size:')
print('=' * 90)
print(funnel_data.to_string(index=False))

Funnel Summary per Entry Size:
   trial_size  total_trial  repeat_shoppers  lapse_shoppers  avg_trial_asp  repeat_rate_%  lapse_rate_%
         本体通常       558524           230293          328231          257.8           41.2          58.8
        詰替超特大       917400           488130          429270          336.7           53.2          46.8
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ       728676           388716          339960          841.7           53.3          46.7
    詰替超ｼﾞｬﾝﾎﾞ         7175             3171            4004          671.6           44.2          55.8
         詰替通常          181               65             116          193.8           35.9          64.1
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ       609745           314853          294892          652.7           51.6          48.4
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ       307514           150983          156531        1,009.4           49.1          50.9
          ｿﾉﾀ        16209             5203           11006        2,094.4           32.1          67.9


In [7]:
# ── Bar chart: trial universe and outcome per size ────────────────────
sizes = funnel_data.sort_values('total_trial', ascending=True)['trial_size'].tolist()

# ── Trial universe bar ────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Bar(
    y=sizes,
    x=funnel_data.set_index('trial_size').loc[sizes, 'total_trial'],
    name='Trial Shoppers',
    orientation='h', marker_color='#6495ED',
    text=funnel_data.set_index('trial_size').loc[sizes, 'total_trial'],
    textposition='inside'
))
fig.update_layout(
    title='Trial Shopper Universe by Entry Size',
    xaxis_title='Number of Shoppers',
    template='plotly_white', height=400,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3)
)
fig.show()

# ── Repeat vs Lapse per size ──────────────────────────────────────────
fig2 = go.Figure()

fig2.add_trace(go.Bar(
    y=sizes,
    x=funnel_data.set_index('trial_size').loc[sizes, 'repeat_shoppers'],
    name='Repeat',
    orientation='h', marker_color='#2E8B57',
    text=[f"{v:,} ({r:.1f}%)" for v, r in zip(
        funnel_data.set_index('trial_size').loc[sizes, 'repeat_shoppers'],
        funnel_data.set_index('trial_size').loc[sizes, 'repeat_rate_%'])],
    textposition='inside'
))
fig2.add_trace(go.Bar(
    y=sizes,
    x=funnel_data.set_index('trial_size').loc[sizes, 'lapse_shoppers'],
    name='Lapse (6-month no return)',
    orientation='h', marker_color='#CC3333',
    text=[f"{v:,} ({r:.1f}%)" for v, r in zip(
        funnel_data.set_index('trial_size').loc[sizes, 'lapse_shoppers'],
        funnel_data.set_index('trial_size').loc[sizes, 'lapse_rate_%'])],
    textposition='inside'
))

fig2.update_layout(
    barmode='stack',
    title='Trial Outcome by Entry Size — Where Do We Lose Shoppers?',
    xaxis_title='Number of Shoppers',
    template='plotly_white', height=400,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3)
)

# Add ASP annotation
for i, size in enumerate(sizes):
    row = funnel_data[funnel_data['trial_size'] == size].iloc[0]
    fig2.add_annotation(x=row['total_trial'] + 50, y=size,
                        text=f"¥{row['avg_trial_asp']:,.0f}",
                        showarrow=False, font=dict(size=11, color='#333'))

fig2.show()

---
## 4. Step 5-2: Alluvial/Sankey — Trial Size → Repeat Size → Lapse Destination

In [8]:
# ── Prepare flow data from journey + Phase 4 lapse destinations ───────
# For repeat shoppers: destination = Ariel + repeat_size
# For lapse shoppers: use Phase 4 aggregated flow data

# Stage 1 flow: trial_size → outcome
flow_stage1 = df_journey.groupby(['trial_size', 'outcome']).agg(
    count=('shopper_key', 'nunique'),
    avg_asp=('trial_asp', 'mean')
).reset_index()

# Stage 2 flow (Repeat branch): outcome → Ariel repeat_size
repeat_flow = df_journey[df_journey['outcome'] == 'Repeat'].groupby('repeat_size').agg(
    count=('shopper_key', 'nunique')
).reset_index()
repeat_flow.columns = ['dest_label', 'count']
repeat_flow['dest_label'] = 'Ariel ' + repeat_flow['dest_label']
repeat_flow['outcome'] = 'Repeat'

# Stage 2 flow (Lapse branch): outcome → destination (from Phase 4)
lapse_dest = p4_dest.copy()
lapse_dest['dest_label'] = lapse_dest['next_sub_brand'] + ' ' + lapse_dest['next_size'].fillna('')
lapse_dest = lapse_dest.rename(columns={'shoppers': 'count'})
# Add category exit
total_lapsed_journey = (df_journey['outcome'] == 'Lapse').sum()
total_tracked_dest = lapse_dest['count'].sum()
category_exit = max(0, total_lapsed_journey - total_tracked_dest)
lapse_dest = pd.concat([
    lapse_dest[['dest_label', 'count']],
    pd.DataFrame([{'dest_label': 'Category Exit', 'count': category_exit}])
], ignore_index=True)
lapse_dest['outcome'] = 'Lapse'

# Simplify to top destinations
top_dests_all = pd.concat([repeat_flow, lapse_dest]).nlargest(15, 'count')['dest_label'].tolist()

flow_stage2 = pd.concat([repeat_flow, lapse_dest], ignore_index=True)
flow_stage2['dest_simplified'] = flow_stage2['dest_label'].apply(
    lambda x: x if x in top_dests_all else 'Other Brands'
)
flow_stage2 = flow_stage2.groupby(['outcome', 'dest_simplified']).agg(count=('count', 'sum')).reset_index()

print('Flow Stage 1 (Trial Size → Outcome):')
print(flow_stage1.to_string(index=False))
print(f'\nFlow Stage 2 (Outcome → Destination) — top entries:')
print(flow_stage2.nlargest(15, 'count').to_string(index=False))

Flow Stage 1 (Trial Size → Outcome):
   trial_size outcome  count  avg_asp
         本体通常   Lapse 328231    255.2
         本体通常  Repeat 230293    261.4
        詰替超特大   Lapse 429270    333.6
        詰替超特大  Repeat 488130    339.5
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   Lapse 339960    856.4
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ  Repeat 388716    828.9
    詰替超ｼﾞｬﾝﾎﾞ   Lapse   4004    667.4
    詰替超ｼﾞｬﾝﾎﾞ  Repeat   3171    676.9
         詰替通常   Lapse    116    201.2
         詰替通常  Repeat     65    180.7
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   Lapse 294892    638.6
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ  Repeat 314853    665.9
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse 156531    979.7
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat 150983  1,040.2
          ｿﾉﾀ   Lapse  11006  2,018.2
          ｿﾉﾀ  Repeat   5203  2,255.5

Flow Stage 2 (Outcome → Destination) — top entries:
outcome        dest_simplified  count
  Lapse          Category Exit 671137
 Repeat            Ariel 詰替超特大 471075
 Repeat    Ariel 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ 426476
  Lapse           Other Brands 390374
 Repeat     Ariel 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ 277807
 Repeat             Ariel 本体通常 21264

In [9]:
# ── Build Sankey flow data ────────────────────────────────────────────
# Stage 1: Trial Size → Outcome (Repeat / Lapse)
# Stage 2: Outcome → Destination (repeat size / lapse destination)

trial_sizes = sorted(df_journey['trial_size'].unique())
outcomes = ['Repeat', 'Lapse']
destinations = sorted(flow_stage2['dest_simplified'].unique())

all_nodes = (
    [f'Trial: {s}' for s in trial_sizes] +
    outcomes +
    [f'→ {d}' for d in destinations]
)
node_idx = {name: i for i, name in enumerate(all_nodes)}

# Build links
sources, targets, values, colors = [], [], [], []

# Stage 1: Trial Size → Outcome
for _, row in flow_stage1.iterrows():
    src = node_idx[f'Trial: {row["trial_size"]}']
    tgt = node_idx[row['outcome']]
    sources.append(src)
    targets.append(tgt)
    values.append(row['count'])
    colors.append('rgba(46,139,87,0.4)' if row['outcome'] == 'Repeat' else 'rgba(204,51,51,0.4)')

# Stage 2: Outcome → Destination
for _, row in flow_stage2.iterrows():
    src = node_idx[row['outcome']]
    tgt = node_idx[f'→ {row["dest_simplified"]}']
    sources.append(src)
    targets.append(tgt)
    values.append(row['count'])
    if ARIEL_GEL in row['dest_simplified'] or 'Ariel' in row['dest_simplified']:
        colors.append('rgba(30,144,255,0.4)')  # Ariel = blue
    elif ATTACK_EX in row['dest_simplified']:
        colors.append('rgba(255,99,71,0.5)')   # Attack = red
    elif 'Exit' in row['dest_simplified']:
        colors.append('rgba(128,128,128,0.3)') # Exit = gray
    else:
        colors.append('rgba(255,165,0,0.3)')   # Other = orange

# Node colors
node_colors = (
    ['#1E90FF'] * len(trial_sizes) +  # Trial sizes = blue
    ['#2E8B57', '#CC3333'] +           # Repeat = green, Lapse = red
    ['#6495ED'] * len(destinations)    # Destinations = light blue
)

fig = go.Figure(go.Sankey(
    node=dict(
        pad=15, thickness=20,
        label=all_nodes,
        color=node_colors
    ),
    link=dict(
        source=sources, target=targets, value=values,
        color=colors
    )
))

fig.update_layout(
    title_text='Shopper Universe Flow: Trial Size → Repeat/Lapse → Destination (Sub-brand + Size)',
    font_size=11, height=700, template='plotly_white'
)
fig.show()

---
## 5. Step 5-3: ASP-Annotated Flow Summary

Overlay the prevailing ASP at each funnel stage to identify which price point to fix.

In [10]:
# ── ASP at each stage per size ────────────────────────────────────────
asp_by_stage = []

for size in trial_sizes:
    size_data = df_journey[df_journey['trial_size'] == size]
    total        = len(size_data)
    repeat_data  = size_data[size_data['outcome'] == 'Repeat']
    lapse_data   = size_data[size_data['outcome'] == 'Lapse']

    asp_by_stage.append({
        'size': size,
        'trial_shoppers': total,
        'avg_trial_asp': size_data['trial_asp'].mean(),
        'repeat_shoppers': len(repeat_data),
        'repeat_rate_%': round(len(repeat_data) / max(total, 1) * 100, 1),
        'avg_repeat_asp': repeat_data['repeat_asp'].mean() if len(repeat_data) > 0 else None,
        'lapse_shoppers': len(lapse_data),
        'lapse_rate_%': round(len(lapse_data) / max(total, 1) * 100, 1),
        'avg_lapse_trial_asp': lapse_data['trial_asp'].mean() if len(lapse_data) > 0 else None,
    })

df_asp_flow = pd.DataFrame(asp_by_stage)

print('=' * 100)
print('ASP-Annotated Funnel: Which SIZE + PRICE POINT needs intervention?')
print('=' * 100)
print(df_asp_flow.to_string(index=False))

print('\n📊 Interpretation Guide:')
print('  - High lapse_rate_% + high avg_trial_asp = price too high for trial conversion')
print('  - Large lapse_shoppers + low avg_trial_asp = price is not the issue, look at product/competitor')
print('  - avg_repeat_asp < avg_trial_asp = shoppers expect discount to repeat → margin risk')

ASP-Annotated Funnel: Which SIZE + PRICE POINT needs intervention?
         size  trial_shoppers  avg_trial_asp  repeat_shoppers  repeat_rate_%  avg_repeat_asp  lapse_shoppers  lapse_rate_%  avg_lapse_trial_asp
         本体通常          558524          257.8           230293           41.2           407.6          328231          58.8                255.2
        詰替超特大          917400          336.7           488130           53.2           427.1          429270          46.8                333.6
詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ          728676          841.7           388716           53.3           766.7          339960          46.7                856.4
    詰替超ｼﾞｬﾝﾎﾞ            7175          671.6             3171           44.2           691.7            4004          55.8                667.4
         詰替通常             181          193.8               65           35.9           299.4             116          64.1                201.2
 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ          609745          652.7           314853        

In [11]:
# ── Bubble chart: size universe with ASP and rates ────────────────────
fig = px.scatter(
    df_asp_flow,
    x='avg_trial_asp',
    y='repeat_rate_%',
    size='trial_shoppers',
    text='size',
    color='lapse_rate_%',
    color_continuous_scale='RdYlGn_r',  # Red = high lapse, Green = low lapse
    labels={
        'avg_trial_asp': 'Average Trial ASP (JPY)',
        'repeat_rate_%': 'Repeat Rate (%)',
        'trial_shoppers': 'Trial Shoppers (bubble size)',
        'lapse_rate_%': 'Lapse Rate (%)'
    },
    title='Strategic Map: Which Size + Price Needs Fixing?<br>(Big bubble = big opportunity, Red = high lapse risk)'
)
fig.update_traces(textposition='top center', marker=dict(sizemin=10))
fig.update_layout(template='plotly_white', height=600)
fig.show()

In [12]:
# ── Export Phase 5 results ────────────────────────────────────────────
output_file = 'phase5_shopper_flow.xlsx'

# Strip timezone info to avoid Excel tz-aware error
for col in funnel_data.select_dtypes(include=['datetimetz']).columns:
    funnel_data[col] = funnel_data[col].dt.tz_localize(None)

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    funnel_data.to_excel(writer, sheet_name='Funnel_by_Size', index=False)
    flow_stage1.to_excel(writer, sheet_name='Flow_TrialSize_to_Outcome', index=False)
    flow_stage2.to_excel(writer, sheet_name='Flow_Outcome_to_Dest', index=False)
    df_asp_flow.to_excel(writer, sheet_name='ASP_Annotated_Funnel', index=False)

print(f'✅ Phase 5 results exported to {output_file}')
print('\n🎯 Next step: Strategy Memo — synthesize findings from all 6 phases.')

✅ Phase 5 results exported to phase5_shopper_flow.xlsx

🎯 Next step: Strategy Memo — synthesize findings from all 6 phases.
